# <center> **PROJECT: Marketplace A/A/B Testing Analysis**

---

#### **Описание:**

У нас есть результаты $A/A/B$-тестирования от одного известного маркетплейса. 

> `sample_a`, `sample_c` — $АА$-группы, `sample_b` — отдельная группа. 

В каждом датасете есть три типа действий пользователей: $0$ — клик, $1$ — просмотр и $2$ — покупка (пользователь просматривает выдачу товаров, кликает на понравившийся товар и совершает покупку).

Маркетплейс ориентируется на следующие метрики:

* *ctr* (отношение кликов к просмотрам товаров);

* *purchase rate* (отношение покупок к просмотрам товаров);

* *gmv* (оборот, сумма произведений количества покупок на стоимость покупки), где считаем $1$ сессию за $1$ точку ($1$ сессия на $1$ пользователя).

---


#### **Постановка задачи:**

В данном проекте необходимо проанализировать группы, выяснить, нет ли проблемы с разъезжанием сплитов и улучшает ли алгоритм $B$ работу маркетплейса.

---


#### **Основные цели:**

* Сформировать набор данных на основе предоставленных источников информации;

* Проанализировать структуру данных и провести их предобработку;

* Провести статистический анализ результатов $A/B$-тестирования.

---


#### **Этапы работы над проектом:**

Проект будет состоять из $4$ частей:

`1.` *Базовый анализ и знакомство с данными*;

`2.` *Очистка данных*;

`3.` *Расчёт метрик*;

`4.` *Финальное заключение по результатам A/A/B-тестирования*.

---

**Импортируем необходимые библиотеки:**

In [1]:
# Для работы с данными
import numpy as np
import pandas as pd

# Для построения графиков
import seaborn as sns
import matplotlib.pyplot as plt

# Для проведения тестов
from scipy import stats
from statsmodels.stats import proportion

**Загрузка данных:**

In [2]:
# Загружаем датасеты
item_prices_data = pd.read_csv('data/item_prices.csv')
sample_a_data = pd.read_csv('data/sample_a.csv')
sample_b_data = pd.read_csv('data/sample_b.csv')
sample_c_data = pd.read_csv('data/sample_c.csv')

=============================================================================================================================================

## <CENTER> **`1.` Базовый анализ и знакомство с данными**

На данном этапе изучим информацию предоставленных данных.

#### $1.1$

Создадим функцию для вывода основной информации по таблице:

In [3]:
# Создадим функцию для вывода информации по датасету
def check_data_inf(data, data_name = None, length1 = None, length2 = None):
    
    # Выводим размеры таблицы
    print(f'\nДанные таблицы "{data_name}" имеют следующую размерность:\n')
    print('Количество строк: {};\nКоличество признаков (столбцов): {}.'.format(data.shape[0], data.shape[1]))
    print('-' * length1)

    # Выведем первые пять строк 
    display(data)
    print('-' * length2)

Выведем информацию по таблицам:

In [4]:
# Создадим список загруженных датасетов
datasets_list = [
    item_prices_data,
    sample_a_data,
    sample_b_data,
    sample_c_data
]

# Создадим список названий загруженных датасетов
datasets_names = [
    'item_prices_data',
    'sample_a_data',
    'sample_b_data',
    'sample_c_data'
]



# Проходимся циклом 'for' по созданномым спискам
for dataset, dataset_name in zip(datasets_list, datasets_names):
    
    # Выводим информацию по таблице при помощи функции 'check_data_inf'
    check_data_inf(
        data = dataset,
        data_name = dataset_name,
        length1 = 40,
        length2 = 40
    )
    print('\n')


Данные таблицы "item_prices_data" имеют следующую размерность:

Количество строк: 1000;
Количество признаков (столбцов): 2.
----------------------------------------


,item_id,item_price
0,338,1501
1,74,647
2,7696,825
3,866,875
4,5876,804
...,...,...
995,3463,1300
996,1573,716
997,610,1748
998,4452,386


----------------------------------------



Данные таблицы "sample_a_data" имеют следующую размерность:

Количество строк: 1188912;
Количество признаков (столбцов): 3.
----------------------------------------


,user_id,item_id,action_id
0,84636,360,1
1,21217,9635,1
2,13445,8590,1
3,38450,5585,1
4,14160,2383,0
...,...,...,...
1188907,22999,2401,1
1188908,23700,4654,0
1188909,18842,3707,1
1188910,32732,9198,1


----------------------------------------



Данные таблицы "sample_b_data" имеют следующую размерность:

Количество строк: 1198438;
Количество признаков (столбцов): 3.
----------------------------------------


,user_id,item_id,action_id
0,118375,4105,1
1,107569,8204,1
2,175990,880,1
3,160582,9568,0
4,123400,4000,1
...,...,...,...
1198433,184760,2645,1
1198434,163544,1371,2
1198435,153724,3788,1
1198436,176308,4164,1


----------------------------------------



Данные таблицы "sample_c_data" имеют следующую размерность:

Количество строк: 1205510;
Количество признаков (столбцов): 3.
----------------------------------------


,user_id,item_id,action_id
0,274623,2863,1
1,265472,343,1
2,242779,6009,0
3,275009,2184,1
4,268104,3134,2
...,...,...,...
1205505,286691,2188,1
1205506,218099,4143,1
1205507,227599,5391,0
1205508,278847,4853,1


----------------------------------------




#### $1.2$

Выведем информацию по признакам:

In [5]:
# Проходимся циклом 'for' по созданному списку
for dataset, dataset_name in zip(datasets_list, datasets_names):
    
    # Выводим информацию
    print(f'\nИнформация по таблице {dataset_name}\n')
    dataset.info()
    print('-' * 40)
    print('\n')


Информация по таблице item_prices_data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   item_id     1000 non-null   int64
 1   item_price  1000 non-null   int64
dtypes: int64(2)
memory usage: 15.8 KB
----------------------------------------



Информация по таблице sample_a_data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1188912 entries, 0 to 1188911
Data columns (total 3 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   user_id    1188912 non-null  int64
 1   item_id    1188912 non-null  int64
 2   action_id  1188912 non-null  int64
dtypes: int64(3)
memory usage: 27.2 MB
----------------------------------------



Информация по таблице sample_b_data

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1198438 entries, 0 to 1198437
Data columns (total 3 columns):
 #   Column     Non-Null Count    

#### $1.3$

Выведем описательные характеристики по признакам:

In [6]:
# Проходимся циклом 'for' по созданным спискам
for dataset, dataset_name in zip(datasets_list, datasets_names):
    
    # Выводим информацию
    print(f'\nИнформация по таблице {dataset_name}\n')
    display(dataset.describe())
    print('-' * 50)
    print('\n')


Информация по таблице item_prices_data



,item_id,item_price
count,1000.000000,1000.000000
mean,4977.474000,1065.759000
std,2868.435617,557.106055
min,21.000000,102.000000
25%,2563.750000,593.250000
50%,4867.000000,1062.000000
75%,7580.000000,1568.250000
max,9994.000000,1998.000000


--------------------------------------------------



Информация по таблице sample_a_data



,user_id,item_id,action_id
count,1.188912e+06,1.188912e+06,1.188912e+06
mean,4.957860e+04,4.970221e+03,8.799995e-01
std,2.920989e+04,2.872307e+03,4.308128e-01
min,2.410000e+02,2.100000e+01,0.000000e+00
25%,2.513300e+04,2.544000e+03,1.000000e+00
50%,4.812400e+04,4.865000e+03,1.000000e+00
75%,7.576600e+04,7.541000e+03,1.000000e+00
max,9.988000e+04,9.994000e+03,2.000000e+00


--------------------------------------------------



Информация по таблице sample_b_data



,user_id,item_id,action_id
count,1.198438e+06,1.198438e+06,1.198438e+06
mean,1.497068e+05,4.970706e+03,9.523805e-01
std,2.867395e+04,2.873114e+03,4.517543e-01
min,1.000380e+05,2.100000e+01,0.000000e+00
25%,1.235220e+05,2.541000e+03,1.000000e+00
50%,1.499440e+05,4.865000e+03,1.000000e+00
75%,1.740590e+05,7.541000e+03,1.000000e+00
max,1.999660e+05,9.994000e+03,2.000000e+00


--------------------------------------------------



Информация по таблице sample_c_data



,user_id,item_id,action_id
count,1.205510e+06,1.205510e+06,1.205510e+06
mean,2.510076e+05,4.971334e+03,8.818898e-01
std,2.820194e+04,2.872129e+03,4.456996e-01
min,2.002250e+05,2.100000e+01,0.000000e+00
25%,2.277640e+05,2.544000e+03,1.000000e+00
50%,2.514010e+05,4.865000e+03,1.000000e+00
75%,2.755120e+05,7.541000e+03,1.000000e+00
max,2.998280e+05,9.994000e+03,2.000000e+00


--------------------------------------------------




=============================================================================================================================================

## <CENTER> **`2.` Очистка данных**

На данном этапе выявим дубликаты и аномалии и, при их наличии, обработаем их.

#### $2.1$

**Проверяем данные на наличие дубликатов:**

In [8]:
print('\n=================== Информация по дубликатам: ===================\n')



# Проходимся циклом 'for' по созданным спискам
for dataset, dataset_name in zip(datasets_list, datasets_names):
    
    # Создаём список признаков
    dupl_columns = list(dataset.columns)

    # Создаём маску дубликатов с помощью метода duplicated() и произведём фильтрацию
    mask = dataset.duplicated(subset = dupl_columns)
    train_duplicates = dataset[mask]
    print(f'\nЧисло найденных дубликатов таблицы "{dataset_name}": {train_duplicates.shape[0]}')
    print('-' * 60)


=================== Информация по дубликатам: ===================


Число найденных дубликатов таблицы "item_prices_data": 0
------------------------------------------------------------

Число найденных дубликатов таблицы "sample_a_data": 0
------------------------------------------------------------

Число найденных дубликатов таблицы "sample_b_data": 0
------------------------------------------------------------

Число найденных дубликатов таблицы "sample_c_data": 0
------------------------------------------------------------


Как видно по результатам, дубликаты во всех таблицах отсутствуют.

#### $2.2$

Проведём проверку, что нет ситуаций, когда происходит покупка/клик без действия просмотра:

In [9]:
# Создаём словарь выборок a, b, c
samples_dict = {
    'A': sample_a_data,
    'B': sample_b_data,
    'C': sample_c_data
}



# Пройдёмся циклом 'for' по созданному словарю
for label, df in samples_dict.items():
    
    # Группировка действий по пользователю и item_id
    grouped_data = df.groupby(['user_id', 'item_id'])['action_id'].agg(set).reset_index()
    
    # Фильтрация: есть клик или покупка, но нет просмотра
    bad_rows = grouped_data[
        (grouped_data['action_id'].apply(lambda x: 0 in x or 2 in x)) &
        (~grouped_data['action_id'].apply(lambda x: 1 in x))
    ]
    
    # Выводим результат
    print(f'\nНарушения в выборке {label}: {len(bad_rows)}')
    print('-' * 25)


Нарушения в выборке A: 0
-------------------------

Нарушения в выборке B: 0
-------------------------

Нарушения в выборке C: 0
-------------------------


Ситуаций, когда происходит покупка/клик без действия просмотра, не обнаружены ни в одной из выборок.

=============================================================================================================================================

## <CENTER> **`3.` Расчёт метрик**

На данном этапе рассчитаем метрики по датасетам, а так же проведём общее сравнение метрик.

#### $3.1$

#### **Расчёт базовых счётчиков.**

Создадим функцию для рассчёта следующих метрик:

  * `ctr` (отношение кликов к просмотрам товаров);
  
  * `purchase rate` (отношение покупок к просмотрам товаров);

  * `gmv` (оборот, сумма произведений количества покупок на стоимость покупки), где считаем $1$ сессию за $1$ точку ($1$ сессия на $1$ пользователя).

In [10]:
# Создаём функцию 'calculate_metrics' для расчета метрик по пользователям
def calculate_metrics(sample_data, item_prices_data):
    
    # Подсчет действий по пользователям
    user_actions = sample_data.groupby(['user_id', 'action_id']).size().unstack(fill_value=0)
    user_actions = user_actions.rename(columns={0: 'clicks', 1: 'views', 2: 'purchases'}).fillna(0)
    
    # Рассчитываем CTR и purchase_rate
    user_actions['ctr'] = user_actions['clicks'] / user_actions['views'].replace(0, np.nan)
    user_actions['purchase_rate'] = user_actions['purchases'] / user_actions['views'].replace(0, np.nan)
    
    # Рассчитываем GMV (сумма цен покупок для каждого пользователя)
    purchases = sample_data[sample_data['action_id'] == 2][['user_id', 'item_id']]
    purchases = purchases.merge(item_prices_data, on='item_id', how='left')
    gmv = purchases.groupby('user_id')['item_price'].sum().reindex(user_actions.index, fill_value=0)
    
    user_actions['gmv'] = gmv
    
    return user_actions

Рассчитываем метрики:

In [11]:
# Рассчитываем метрики для каждой группы
metrics_a = calculate_metrics(sample_a_data, item_prices_data)
metrics_b = calculate_metrics(sample_b_data, item_prices_data)
metrics_c = calculate_metrics(sample_c_data, item_prices_data)

Сравним агрегированные значения метрик:

In [12]:
# Выводим агрегированные значения метрик для сравнения
print('\n============= Агрегированные значения метрик =============')



# Пройдёмся циклом 'for' по рассчитанным метрикам
for metric in ['ctr', 'purchase_rate', 'gmv']:
    print(f'\nМетрика: {metric}')
    print('-' * 50)
    print(f'Среднее значение для группы A: {round(metrics_a[metric].mean(), 4)}')
    print(f'Среднее значение для группы C: {round(metrics_c[metric].mean(), 4)}')
    print(f'Среднее значение для группы B: {round(metrics_b[metric].mean(), 4)}')
    print('-' * 50)


============= Агрегированные значения метрик =============

Метрика: ctr
--------------------------------------------------
Среднее значение для группы A: 0.2
Среднее значение для группы C: 0.21
Среднее значение для группы B: 0.16
--------------------------------------------------

Метрика: purchase_rate
--------------------------------------------------
Среднее значение для группы A: 0.05
Среднее значение для группы C: 0.06
Среднее значение для группы B: 0.1
--------------------------------------------------

Метрика: gmv
--------------------------------------------------
Среднее значение для группы A: 53273.2701
Среднее значение для группы C: 63998.9034
Среднее значение для группы B: 106473.2711
--------------------------------------------------


Проводим сравнение $A/A$-групп:

In [13]:
# Задаём уровень значимости
alpha_comparison = 0.05



# Сравнение A/A-групп (sample_a и sample_c) для проверки разъезжания сплитов
print('\n============================ Сравнение A/A-групп (sample_a и sample_c) ============================')

for metric in ['ctr', 'purchase_rate', 'gmv']:
    print(f'\n\nСравнение метрики: {metric}')
    print('-' * 35)
    data_a = metrics_a[metric].dropna()
    data_c = metrics_c[metric].dropna()
    
    if len(data_a) >= 3 and len(data_c) >= 3:
        
        # Предварительный тест Шапиро-Уилка для выбора теста сравнения
        shapiro_a = stats.shapiro(data_a) if len(data_a) >= 3 else (None, 0)
        shapiro_c = stats.shapiro(data_c) if len(data_c) >= 3 else (None, 0)
        
        if shapiro_a[1] > 0.01 and shapiro_c[1] > 0.01:
            stat, p_value = stats.ttest_ind(data_a, data_c)
            test_name = 't-тест'
        else:
            stat, p_value = stats.mannwhitneyu(data_a, data_c)
            test_name = 'U-тест Манна-Уитни'
        print(f'{test_name}: p-value = {round(p_value, 4)}')
        
        if p_value > alpha_comparison:
            print(f'A/A-группы (A и C) схожи по метрике {metric}. Проблем с разъезжанием сплитов нет.')
        else:
            print(f'A/A-группы (A и C) различаются по метрике {metric}. Возможны проблемы с разъезжанием сплитов.')
    else:
        print(f'Недостаточно данных для сравнения метрики {metric} между группами A и C.')
        
    print('-' * 97)


============================ Сравнение A/A-групп (sample_a и sample_c) ============================


Сравнение метрики: ctr
-----------------------------------
t-тест: p-value = 0.0
A/A-группы (A и C) различаются по метрике ctr. Возможны проблемы с разъезжанием сплитов.
-------------------------------------------------------------------------------------------------


Сравнение метрики: purchase_rate
-----------------------------------
U-тест Манна-Уитни: p-value = 0.0
A/A-группы (A и C) различаются по метрике purchase_rate. Возможны проблемы с разъезжанием сплитов.
-------------------------------------------------------------------------------------------------


Сравнение метрики: gmv
-----------------------------------
t-тест: p-value = 0.0
A/A-группы (A и C) различаются по метрике gmv. Возможны проблемы с разъезжанием сплитов.
-------------------------------------------------------------------------------------------------


Проводим сравнение группы $B$ с объединенными $A/A$-группами:

In [14]:
print('\n============================ Сравнение группы B с объединенными A/A-группами ============================')

# Задаём уровень значимости 'alpha'           
alpha = 0.01



# Сравнение группы B с объединенными A/A-группами
for metric in ['ctr', 'purchase_rate', 'gmv']:
    print(f'\n\nСравнение метрики: {metric}')
    print('-' * 35)
    
    # Удаляем пропуски
    data_a = metrics_a[metric].dropna()
    data_c = metrics_c[metric].dropna()
    data_b = metrics_b[metric].dropna()
    
    # Объединяем датасеты 'data_a' и 'data_c'
    data_aa = pd.concat([data_a, data_c]).dropna()
    if len(data_aa) >= 3 and len(data_b) >= 3:
        
        # Предварительный тест Шапиро-Уилка для выбора теста сравнения
        shapiro_aa = stats.shapiro(data_aa) if len(data_aa) >= 3 else (None, 0)
        shapiro_b = stats.shapiro(data_b) if len(data_b) >= 3 else (None, 0)
        
        if shapiro_aa[1] > alpha and shapiro_b[1] > alpha:
            stat, p_value = stats.ttest_ind(data_aa, data_b)
            test_name = 't-тест'
        else:
            stat, p_value = stats.mannwhitneyu(data_aa, data_b)
            test_name = 'U-тест Манна-Уитни'
        print(f'{test_name}: p-value = {round(p_value, 4)}')
        
        if p_value <= alpha_comparison:
            print(f'Группа B статистически отличается от A/A-групп по метрике {metric}.')
            aa_mean = data_aa.mean()
            b_mean = data_b.mean()
            
            if b_mean > aa_mean:
                print(f'Алгоритм B улучшает метрику {metric} (среднее B: {round(b_mean, 2)} > среднее A/A: {round(aa_mean, 2)}).')
            else:
                print(f'Алгоритм B ухудшает метрику {metric} (среднее B: {round(b_mean, 2)} < среднее A/A: {round(aa_mean, 2)}).')
                
        else:
            print(f'Группа B не отличается от A/A-групп по метрике {metric}.')
            
    else:
        print(f'Недостаточно данных для сравнения метрики {metric} между группой B и A/A-группами.')
        
    print('-' * 80)


============================ Сравнение группы B с объединенными A/A-группами ============================


Сравнение метрики: ctr
-----------------------------------
t-тест: p-value = 0.0
Группа B статистически отличается от A/A-групп по метрике ctr.
Алгоритм B ухудшает метрику ctr (среднее B: 0.16 < среднее A/A: 0.2).
--------------------------------------------------------------------------------


Сравнение метрики: purchase_rate
-----------------------------------
U-тест Манна-Уитни: p-value = 0.0
Группа B статистически отличается от A/A-групп по метрике purchase_rate.
Алгоритм B улучшает метрику purchase_rate (среднее B: 0.1 > среднее A/A: 0.05).
--------------------------------------------------------------------------------


Сравнение метрики: gmv
-----------------------------------
U-тест Манна-Уитни: p-value = 0.0
Группа B статистически отличается от A/A-групп по метрике gmv.
Алгоритм B улучшает метрику gmv (среднее B: 106473.27 > среднее A/A: 58630.7).
--------------------

#### $3.2$

#### **Проводим тест для нормальности распределения**

Проводим тест `Шапиро-Уилка`:

In [15]:
print('\n============================ Тест Шапиро-Уилка для нормальности распределения ============================')

# Задаём уровень значимости
alpha_shapiro = 0.01



# Пройдёмся циклом 'for' по созданным метрикам
for metric in ['ctr', 'purchase_rate', 'gmv']:
    print(f'\n\nМетрика: {metric}')
    print('-' * 25)
    print()
    
    # Проходимся циклом 'for' по группам
    for group_name, group_data in [('A', metrics_a), ('B', metrics_b), ('C', metrics_c)]:
        data = group_data[metric].dropna()
        
        # Если длинна 'data' больше или равно 3, проводим тест
        if len(data) >= 3:
            shapiro_result = stats.shapiro(data)
            print(f'p-value группы {group_name}: {round(shapiro_result.pvalue, 2)}')
            
            if shapiro_result.pvalue <= alpha_shapiro:
                print(f'Мы отвергаем нулевую гипотезу. Распределение {metric} в группе {group_name} отличается от нормального.')
            else:
                print(f'Мы принимаем нулевую гипотезу. Распределение {metric} в группе {group_name} является нормальным.')
                
        else:
            print(f'Недостаточно данных для теста Шапиро-Уилка в группе {group_name} для метрики {metric}.')
        print('-' * 100)
    print('\n')


============================ Тест Шапиро-Уилка для нормальности распределения ============================


Метрика: ctr
-------------------------

p-value группы A: 0.02
Мы принимаем нулевую гипотезу. Распределение ctr в группе A является нормальным.
----------------------------------------------------------------------------------------------------
p-value группы B: 0.33
Мы принимаем нулевую гипотезу. Распределение ctr в группе B является нормальным.
----------------------------------------------------------------------------------------------------
p-value группы C: 0.17
Мы принимаем нулевую гипотезу. Распределение ctr в группе C является нормальным.
----------------------------------------------------------------------------------------------------




Метрика: purchase_rate
-------------------------

p-value группы A: 0.01
Мы отвергаем нулевую гипотезу. Распределение purchase_rate в группе A отличается от нормального.
-------------------------------------------------------------

#### $3.3$

#### **Проводим тест равенства долей для `A` и `C` групп.**

Создадим функцию для проведения теста равенства долей:

In [16]:
# Создаём функцию 'get_proportions'
def get_proportions(sample_data, action_type):
    
    actions = sample_data[sample_data['action_id'] == action_type].groupby('user_id').size()
    views = sample_data[sample_data['action_id'] == 1].groupby('user_id').size()
    counts = actions.reindex(views.index, fill_value=0).values
    totals = views.values
    
    return counts.sum(), totals.sum()

Проводим $z$-тест для метрики `CTR`:

In [31]:
# Тест равенства долей
print('\n============ Тест равенства долей для A и C групп ============\n\n')



# Задаём уровень значимости
alpha_comparison = 0.05

# CTR (action_id = 0 / action_id = 1)
print('\nМетрика: CTR')
print('-' * 20)
print()

clicks_a, views_a = get_proportions(sample_a_data, 0)
clicks_c, views_c = get_proportions(sample_c_data, 0)


if views_a > 0 and views_c > 0:
    
    stat, p_value = proportion.proportions_ztest([clicks_a, clicks_c], [views_a, views_c])
    print(f'Z-тест для CTR: p-value = {round(p_value, 4)}')
    print(f'CTR группы A: {round(clicks_a / views_a, 4) if views_a > 0 else "N/A"}')
    print(f'CTR группы C: {round(clicks_c / views_c, 4) if views_c > 0 else "N/A"}')
    
    if p_value > alpha_comparison:
        print('A/A-группы (A и C) схожи по CTR. Проблем с разъезжанием сплитов нет.')
    else:
        print('A/A-группы (A и C) различаются по CTR. Возможны проблемы с разъезжанием сплитов.')
        
else:
    print('Недостаточно данных для теста равенства долей по CTR.')
print('-' * 80)


============ Тест равенства долей для A и C групп ============



Метрика: CTR
--------------------

Z-тест для CTR: p-value = 0.0
CTR группы A: 0.2
CTR группы C: 0.21
A/A-группы (A и C) различаются по CTR. Возможны проблемы с разъезжанием сплитов.
--------------------------------------------------------------------------------


Проводим $z$-тест для метрики `Purchase Rate`:

In [32]:
# Тест равенства долей
print('\n======================== Тест равенства долей для A и C групп ========================\n\n')



# Purchase Rate (action_id = 2 / action_id = 1)
print('\nМетрика: Purchase Rate')
print('-' * 25)
print()

purchases_a, views_a = get_proportions(sample_a_data, 2)
purchases_c, views_c = get_proportions(sample_c_data, 2)

if views_a > 0 and views_c > 0:
    
    stat, p_value = proportion.proportions_ztest([purchases_a, purchases_c], [views_a, views_c])
    print(f'Z-тест для Purchase Rate: p-value = {round(p_value, 4)}')
    print(f'Purchase Rate группы A: {round(purchases_a / views_a, 4) if views_a > 0 else "N/A"}')
    print(f'Purchase Rate группы C: {round(purchases_c / views_c, 4) if views_c > 0 else "N/A"}')
    
    if p_value > alpha_comparison:
        print('A/A-группы (A и C) схожи по Purchase Rate. Проблем с разъезжанием сплитов нет.')
    else:
        print('A/A-группы (A и C) различаются по Purchase Rate. Возможны проблемы с разъезжанием сплитов.')
        
else:
    print('Недостаточно данных для теста равенства долей по Purchase Rate.')
print('-' * 90)


======================== Тест равенства долей для A и C групп ========================



Метрика: Purchase Rate
-------------------------

Z-тест для Purchase Rate: p-value = 0.0
Purchase Rate группы A: 0.05
Purchase Rate группы C: 0.06
A/A-группы (A и C) различаются по Purchase Rate. Возможны проблемы с разъезжанием сплитов.
------------------------------------------------------------------------------------------


Теперь проведём $t$-тест для `GMV`:

In [30]:
# t-тест для GMV
print('\n=============== Сравнение GMV для A/A-групп (sample_a и sample_c) ===============\n\n')



# GMV (t-тест, так как оба распределения нормальны)
print('\nМетрика: GMV')
print('-' * 13)
print()

data_a = metrics_a['gmv'].dropna()
data_c = metrics_c['gmv'].dropna()

if len(data_a) >= 3 and len(data_c) >= 3:
    
    stat, p_value = stats.ttest_ind(data_a, data_c)
    print(f't-тест для GMV: p-value = {round(p_value, 4)}')
    print(f'Среднее GMV группы A: {round(data_a.mean(), 2)}')
    print(f'Среднее GMV группы C: {round(data_c.mean(), 2)}')
    
    if p_value > alpha_comparison:
        print('A/A-группы (A и C) схожи по GMV. Проблем с разъезжанием сплитов нет.')
    else:
        print('A/A-группы (A и C) различаются по GMV. Возможны проблемы с разъезжанием сплитов.')
        
else:
    print('Недостаточно данных для сравнения GMV между группами A и C.')
print('-' * 80)


=============== Сравнение GMV для A/A-групп (sample_a и sample_c) ===============



Метрика: GMV
-------------

t-тест для GMV: p-value = 0.0
Среднее GMV группы A: 53273.27
Среднее GMV группы C: 63998.9
A/A-группы (A и C) различаются по GMV. Возможны проблемы с разъезжанием сплитов.
--------------------------------------------------------------------------------


**Вывод:**

Все три метрики (`CTR`, `Purchase Rate`, `GMV`) показывают статистически значимые различия между $A/A$-группами (`p-value = 0.0` для всех). Это означает, что группы $A$ и $C$ не эквивалентны, несмотря на то, что они должны быть идентичными в рамках $A/A$-теста.

#### $3.4$

#### **Проводим тест равенства долей для `A` и `B` групп.**

Проводим $z$-тест для метрики `CTR`:

In [34]:
# Тест равенства долей
print('\n============ Тест равенства долей для A и B групп ============\n\n')



# Задаём уровень значимости
alpha_comparison = 0.05

# CTR (action_id = 0 / action_id = 1)
print('\nМетрика: CTR')
print('-' * 20)
print()

clicks_a, views_a = get_proportions(sample_a_data, 0)
clicks_b, views_b = get_proportions(sample_b_data, 0)

if views_a > 0 and views_b > 0:
    
    stat, p_value = proportion.proportions_ztest([clicks_a, clicks_b], [views_a, views_b])
    print(f'Z-тест для CTR: p-value = {round(p_value, 4)}')
    print(f'CTR группы A: {round(clicks_a / views_a, 4) if views_a > 0 else "N/A"}')
    print(f'CTR группы B: {round(clicks_b / views_b, 4) if views_b > 0 else "N/A"}')
    
    if p_value <= alpha_comparison:
        print(f'Группа B отличается от группы A по CTR.')
        
        if (clicks_b / views_b if views_b > 0 else 0) > (clicks_a / views_a if views_a > 0 else 0):
            print('Алгоритм B улучшает CTR.')
        else:
            print('Алгоритм B ухудшает CTR.')
            
    else:
        print('Группа B не отличается от группы A по CTR.')
        
else:
    print('Недостаточно данных для теста равенства долей по CTR.')
print('-' * 40)


============ Тест равенства долей для A и B групп ============



Метрика: CTR
--------------------

Z-тест для CTR: p-value = 0.0
CTR группы A: 0.2
CTR группы B: 0.16
Группа B отличается от группы A по CTR.
Алгоритм B ухудшает CTR.
----------------------------------------


Проводим $z$-тест для метрики `Purchase Rate`:

In [36]:
# Тест равенства долей
print('\n============ Тест равенства долей для A и B групп ============\n\n')



# Purchase Rate (action_id = 2 / action_id = 1)
print('\nМетрика: Purchase Rate')
print('-' * 25)
print()

purchases_a, views_a = get_proportions(sample_a_data, 2)
purchases_b, views_b = get_proportions(sample_b_data, 2)

if views_a > 0 and views_b > 0:
    stat, p_value = proportion.proportions_ztest([purchases_a, purchases_b], [views_a, views_b])
    print(f'Z-тест для Purchase Rate: p-value = {round(p_value, 4)}')
    print(f'Purchase Rate группы A: {round(purchases_a / views_a, 4) if views_a > 0 else "N/A"}')
    print(f'Purchase Rate группы B: {round(purchases_b / views_b, 4) if views_b > 0 else "N/A"}')
    
    if p_value <= alpha_comparison:
        print(f'Группа B отличается от группы A по Purchase Rate.')
        
        if (purchases_b / views_b if views_b > 0 else 0) > (purchases_a / views_a if views_a > 0 else 0):
            print('Алгоритм B улучшает Purchase Rate.')
        else:
            print('Алгоритм B ухудшает Purchase Rate.')
            
    else:
        print('Группа B не отличается от группы A по Purchase Rate.')
        
else:
    print('Недостаточно данных для теста равенства долей по Purchase Rate.')
print('-' * 50)


============ Тест равенства долей для A и B групп ============



Метрика: Purchase Rate
-------------------------

Z-тест для Purchase Rate: p-value = 0.0
Purchase Rate группы A: 0.05
Purchase Rate группы B: 0.1
Группа B отличается от группы A по Purchase Rate.
Алгоритм B улучшает Purchase Rate.
--------------------------------------------------


Теперь проведём $t$-тест для `GMV`:

In [39]:
# t-тест для GMV
print('\n=============== Сравнение GMV для групп A и B ===============\n\n')



# GMV (t-тест, так как оба распределения нормальны)
print('\nМетрика: GMV')
print('-' * 13)
print()

data_a = metrics_a['gmv'].dropna()
data_b = metrics_b['gmv'].dropna()

if len(data_a) >= 3 and len(data_b) >= 3:
    stat, p_value = stats.ttest_ind(data_a, data_b)
    print(f't-тест для GMV: p-value = {round(p_value, 4)}')
    print(f'Среднее GMV группы A: {round(data_a.mean(), 2)}')
    print(f'Среднее GMV группы B: {round(data_b.mean(), 2)}')
    
    if p_value <= alpha_comparison:
        print(f'Группа B отличается от группы A по GMV.')
        
        if data_b.mean() > data_a.mean():
            print('Алгоритм B улучшает GMV.')
        else:
            print('Алгоритм B ухудшает GMV.')
            
    else:
        print('Группа B не отличается от группы A по GMV.')
        
else:
    print('Недостаточно данных для сравнения GMV между группами A и B.')
print('-' * 43)


=============== Сравнение GMV для групп A и B ===============



Метрика: GMV
-------------

t-тест для GMV: p-value = 0.0
Среднее GMV группы A: 53273.27
Среднее GMV группы B: 106473.27
Группа B отличается от группы A по GMV.
Алгоритм B улучшает GMV.
-------------------------------------------


**Вывод:**

Алгоритм B демонстрирует улучшение по двум ключевым метрикам:

  * **Purchase Rate**: Увеличение с $5$% до $10$% указывает на более высокую конверсию просмотров в покупки.

  * **GMV**: Среднее GMV в группе B ($106473.27$) почти вдвое выше, чем в группе A ($53273.27$), что говорит о значительном увеличении оборота.

Однако алгоритм B ухудшает CTR (снижение с $20$% до $16$%), что может указывать на меньшую привлекательность товаров или выдачи для пользователей, несмотря на рост покупок.

=============================================================================================================================================

## <CENTER> **`4.` Финальное заключение по результатам A/A/B-тестирования**

#### **Проблемы с разъезжанием сплитов (A/A-группы: sample_a и sample_c)**

Тест равенства долей и t-тест для GMV показали статистически значимые различия между A/A-группами по всем метрикам (`CTR`, `Purchase Rate`, `GMV`) с `p-value = 0.0` (при `alpha = 0.05`). Это указывает на некорректное сплитование:

  * `CTR`: Различие между $20$% (A) и $21$% (C).
  * `Purchase Rate`: Различие между $5$% (A) и $6$% (C).
  * `GMV`: Различие между средним значением $53273.27$ (A) и $63998.9$ (C).


**Вывод:** A/A-группы, которые должны быть идентичными, демонстрируют значительные различия, что свидетельствует о проблемах в системе сплитования. 

Возможные причины:

  * Дисбаланс характеристик пользователей (например, возраст, регион, история покупок).
  * Технические ошибки в механизме рандомизации.
  * Внешние факторы (например, разные временные интервалы или акции).

Эти проблемы делают результаты $A/A/B$-теста ненадежными, так как различия между группами $A$ и $B$ могут быть обусловлены не только алгоритмом $B$, но и некорректным сплитованием.